# Training on images and spectra together

This notebook trains a PyTorch model on a multimodal HATS catalog, where every row holds both
an image and a spectrum of the same galaxy. The catalog is a crossmatch of Legacy Survey DR10 images
with DESI EDR spectra, hosted on Hugging Face.

The model is not meant to do real science. It shows how the different column types of a catalog
reach a model:

- **Images** are fixed-shape tensor columns (`tensor[float, (160, 160)]`), one per band.
- **Spectra** are a nested column, with one row of `flux`, `lambda`, `ivar`, ... per object.

Data is streamed with `CatalogDataset`, and several PyTorch workers load batches in parallel while
the model trains. We train once while streaming the catalog from Hugging Face and once from a copy
on local disk, and compare how fast data reaches the GPU in each case.

This notebook is pre-executed because it reads data over the network and is meant for a GPU.

In [ ]:
import multiprocessing
import sys
import time
from pathlib import Path

import lsdb
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.nn import functional as F

from lsdb_ml import CatalogDataLoader, CatalogDataset

## Settings

`SOURCES` chooses where the data comes from, and each source gets its own training run:

- `"remote"` streams the catalog from Hugging Face in every epoch.
- `"local"` reads a copy on local disk. The partitions we use are saved to `LOCAL_PATH` the first
  time (about 2.5 GB), unless a complete copy is already there.

Use `["remote"]`, `["local"]` or both. If you change `N_PARTITIONS`, delete `LOCAL_PATH` so the
copy is written again.

`PRELOAD` lists the modules that the workers' fork server imports once, so new workers start
faster (see "Stream batches with several workers"). The fork server keeps its preloads, so restart
the kernel after changing `SOURCES` or `PRELOAD`.

In [ ]:
SOURCES = ["remote", "local"]
REMOTE_URL = "hf://datasets/brantd/multimodal_hats_example"
LOCAL_PATH = "multimodal_hats_local"
N_PARTITIONS = 32  # of 548, so the notebook runs in minutes
NUM_WORKERS = 4  # PyTorch workers that load batches
EPOCHS = 3

# On macOS, workers that read over HTTPS crash if LSDB was preloaded, so preload only torch there.
PRELOAD = ["torch", "lsdb", "lsdb_ml", "nested_pandas"]
if sys.platform == "darwin" and "remote" in SOURCES:
    PRELOAD = ["torch"]

## Load the catalog

We open only the columns we need, the `g` and `r` band images and the spectra, and keep the first
`N_PARTITIONS` partitions. For `"local"`, the same partitions are written to disk and opened from
there.

In [ ]:
columns = ["image_flux_g", "image_flux_r", "spectrum"]
catalogs = {"remote": lsdb.open_catalog(REMOTE_URL, columns=columns).partitions[:N_PARTITIONS]}
if "local" in SOURCES:
    # collection.properties is written last, so it exists only if a previous write finished.
    if not Path(LOCAL_PATH, "collection.properties").exists():
        catalogs["remote"].write_catalog(LOCAL_PATH, overwrite=True)
    catalogs["local"] = lsdb.open_catalog(LOCAL_PATH, columns=columns)
catalogs["remote"]

## Turn rows into tensors

The default `collate_catalog` handles flat and nested numeric columns, but not tensor columns, so
we write a `collate_fn`. It receives a `NestedFrame` with one batch of rows and returns tensors:

- `to_stack()` turns a tensor column into one `(batch, 160, 160)` array. We stack the two bands
  into channels.
- Every DESI spectrum is sampled on the same wavelength grid, so the flat `flux` values reshape into
  a `(batch, n_wavelengths)` array.

In [ ]:
def collate(frame):
    image = np.stack([frame["image_flux_g"].to_stack(), frame["image_flux_r"].to_stack()], axis=1)
    spectrum = frame["spectrum"].nest["flux"].to_numpy(dtype=np.float32).reshape(len(frame), -1)
    return {"image": torch.from_numpy(image), "spectrum": torch.tensor(spectrum)}


example = collate(catalogs[SOURCES[0]].partitions[0].compute())
{name: tensor.shape for name, tensor in example.items()}

## Stream batches with several workers

With `num_workers > 0`, each PyTorch worker streams its own share of the partitions with dask's
synchronous scheduler, so no dask client is needed.

We start the workers with `forkserver`, because forking the notebook process directly (the default
on Linux) can hang. The dataset, including our `collate` function, is sent to each worker with
`cloudpickle`, so functions defined in a notebook work.

Every epoch starts new workers. The fork server imports the modules in `PRELOAD` once, and each
worker is forked from it, so workers do not import torch and LSDB themselves.

> **macOS and HTTPS.** On macOS, workers that read over HTTPS (e.g. `hf://`) segfault on their first
> request if LSDB, nested-pandas or pandas was preloaded, because the macOS certificate check
> (`truststore`) is not fork-safe. So the settings cell preloads only torch on macOS when `SOURCES`
> includes `"remote"`. Reading a local copy is not affected.

In [ ]:
context = multiprocessing.get_context("forkserver")
context.set_forkserver_preload(PRELOAD)


def make_loader(catalog):
    dataset = CatalogDataset(catalog, batch_size=64, seed=0, drop_last=True, collate_fn=collate)
    return CatalogDataLoader(dataset, num_workers=NUM_WORKERS, multiprocessing_context=context)

## A two-tower model

Each modality gets its own encoder. The images go through a standard ResNet-18 from torchvision,
with its first layer changed to take 2 bands instead of 3 colors. The spectra go through a small 1D
CNN. Both map to a shared 64-dimensional embedding. A `BatchNorm` at the input of each encoder takes
care of the very different flux scales of the two.

We train it contrastively, as in CLIP or AstroCLIP: in a batch of `B` objects, the image of each
object should be closer to its own spectrum than to the other `B - 1` spectra. No labels are
needed.

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        resnet = torchvision.models.resnet18(num_classes=dim)
        resnet.conv1 = nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False)  # 2 bands, not RGB
        self.image = nn.Sequential(nn.BatchNorm2d(2), resnet)
        self.spectrum = nn.Sequential(
            nn.BatchNorm1d(1),
            nn.Conv1d(1, 16, 9, stride=8),
            nn.ReLU(),
            nn.Conv1d(16, 32, 9, stride=8),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, dim),
        )

    def forward(self, image, spectrum):
        return F.normalize(self.image(image), dim=1), F.normalize(self.spectrum(spectrum[:, None]), dim=1)

## Train

Each source trains a fresh model for `EPOCHS` epochs, so the runs differ only in where the data
comes from. `CatalogDataLoader` gives each epoch a new shuffle order. Besides the loss, we track
how often an image's nearest spectrum in the batch is its own; chance is `1 / 64`.

We also log how long each batch took, for the throughput report: `wait_s` is the time the loop
waited for the loader, and `step_s` the time spent in the training step.

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)


def new_model():
    torch.manual_seed(0)
    model = TwoTower().to(device)
    return model, torch.optim.Adam(model.parameters(), lr=1e-3)


def train_step(model, optimizer, batch):
    image_emb, spectrum_emb = model(batch["image"].to(device), batch["spectrum"].to(device))
    logits = image_emb @ spectrum_emb.T / 0.1
    target = torch.arange(len(logits), device=device)
    loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # .item() waits for the GPU, so timing a call to train_step includes its GPU work.
    return loss.item(), (logits.argmax(1) == target).float().mean().item()


def train(source):
    """Train a fresh model on one source; returns one row of timings per batch."""
    model, optimizer = new_model()
    loader = make_loader(catalogs[source])
    log = []
    for epoch in range(EPOCHS):
        last = time.perf_counter()
        for batch in loader:
            loaded = time.perf_counter()
            loss, matched = train_step(model, optimizer, batch)
            done = time.perf_counter()
            log.append(
                dict(
                    source=source,
                    epoch=epoch,
                    wait_s=loaded - last,
                    step_s=done - loaded,
                    loss=loss,
                    matched=matched,
                )
            )
            last = done
        epoch_log = pd.DataFrame(log).query("epoch == @epoch")
        print(
            f"{source}, epoch {epoch}: loss {epoch_log.loss.mean():.3f}, matched {epoch_log.matched.mean():.1%}"
        )
    return pd.DataFrame(log)


log = pd.concat([train(source) for source in SOURCES], ignore_index=True)

## Throughput report

Is the GPU waiting on LSDB, or LSDB on the GPU? Three numbers tell us:

1. **Data rate.** Objects per second that the loader delivered during training, and the size of
   those tensors in MB per second. (This is the in-memory size, not the compressed bytes read.)
2. **GPU ceiling.** Objects per second the training step can process when the data is already in
   memory, measured by repeating the step on one batch. It does not depend on the source.
3. **GPU-busy fraction.** The share of wall time the loop spent in the training step. The rest was
   spent waiting for the next batch.

If the GPU-busy fraction is close to 1, the model is the bottleneck and faster loading would not
help. If it is low, the GPU sits idle and training runs only as fast as the data arrives.

The step time includes copying the batch to the device and some Python overhead, so the busy
fraction is an upper bound on how busy the GPU itself was. Without a GPU, the model trains on the
CPU, and "GPU" in this section means the CPU: the ceiling is lower, and the training step shares the
cores with the loader workers.

In [ ]:
# One batch, loaded in this process without workers.
batch = next(iter(CatalogDataset(catalogs[SOURCES[-1]], batch_size=64, collate_fn=collate)))
model, optimizer = new_model()
train_step(model, optimizer, batch)  # warm up
start = time.perf_counter()
for _ in range(20):
    train_step(model, optimizer, batch)
gpu_ceiling = 20 * len(batch["image"]) / (time.perf_counter() - start)
print(f"GPU ceiling: {gpu_ceiling:,.0f} objects/s on {device}")

Each epoch's numbers come from the timings logged during training, split in two parts:

- **Startup.** `startup_s` is the wait for the epoch's first batch, while the workers start and read
  their first partitions. `epoch_s` is the whole epoch, including it.
- **Steady state.** `objects_per_s`, `MB_per_s`, `gpu_busy` and `gpu_headroom` cover the rest of the
  epoch, once the workers are running. `gpu_headroom` is how many times faster the GPU could train
  than the data arrived.

In [ ]:
mb_per_object = sum(tensor[0].nbytes for tensor in batch.values()) / 1e6
report = log.groupby(["source", "epoch"], sort=False).agg(
    batches=("step_s", "size"),
    startup_s=("wait_s", "first"),
    wait_s=("wait_s", "sum"),
    step_s=("step_s", "sum"),
)
report["epoch_s"] = report.wait_s + report.step_s
steady_s = report.epoch_s - report.startup_s  # time after the first batch arrived
report["objects_per_s"] = report.batches * len(batch["image"]) / steady_s
report["MB_per_s"] = report.objects_per_s * mb_per_object
report["gpu_busy"] = report.step_s / steady_s
report["gpu_headroom"] = gpu_ceiling / report.objects_per_s
print(f"{NUM_WORKERS} workers, preloaded {PRELOAD}")
report[["batches", "epoch_s", "startup_s", "objects_per_s", "MB_per_s", "gpu_busy", "gpu_headroom"]].round(2)

### Reading the results

The `gpu_busy` column says which side is the bottleneck:

- **Close to 1: the model is the bottleneck.** Faster loading would not help; a larger batch, mixed
  precision or a faster GPU would.
- **Well below 1: the data is the bottleneck.** The GPU sits idle between batches, and training runs
  only as fast as `objects_per_s`. `gpu_headroom` says how much faster it could go.

**Remote vs local.** The two runs use the same partitions, model and workers, so the difference
between them is what reading over the network costs. Each partition holds about 100 MB of images
and spectra. Reading from Hugging Face, the data rate is set by how fast those can be downloaded,
and the GPU waits on data. Reading the local copy removes that limit, and the balance then depends
on the model.

**Worker startup** is a fixed cost per epoch, so compare `startup_s` with `epoch_s`. A worker's
first batch waits until its first partition has been read and converted, and with short epochs this
can take a large share of each epoch even while the steady state keeps the GPU busy.
`CatalogDataLoader(..., persistent_workers=True)` keeps the workers between epochs, but then every
epoch repeats the same shuffle order.

To move the bottleneck toward the GPU:

- **Read from faster storage.** A local copy of the catalog (e.g. on a cluster's file system)
  removes the network limit.
- **Then add workers.** Once reads are fast, decompression and `collate` become the CPU cost, and
  that work spreads across `NUM_WORKERS`.
- **The model sets the other side.** A heavier model per object raises `gpu_busy` without any
  change to the data side; a lighter one lowers it.